<a href="https://colab.research.google.com/github/maybebool/boe-financial-analysis/blob/inessa/inessa_llm_analysis_23_sept.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

if torch.cuda.is_available():
    raise RuntimeError(
        "A GPU runtime is connected. Switch Hardware accelerator "
        "to CPU before running cells 0–35."
    )

CPU_DEVICE = torch.device("cpu")

print("Runtime correctly configured for:", CPU_DEVICE)
print("CUDA available:", torch.cuda.is_available())

Runtime correctly configured for: cpu
CUDA available: False


In [2]:
# project setup - run first, do not edit except NAME
NAME = "inessa"  # <<< your name

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
    # keep the throwaway Colab clone current; never touch a local working repo
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
    subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
                   check=False)
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}  data {DATA}")

Mounted at /content/drive
ok  python 3.13.15  pandas 2.2.3  data /content/drive/MyDrive/boe-data


In [3]:
print("exports:", loading.exports(DATA))
print("files:  ", loading.files(DATA, loading.LATEST))
print("banks:  ", loading.firms(DATA, loading.LATEST))

exports: ['2026-09-13', '2026-09-23', 'Never ever upload or change something in the folder above']
files:   []
banks:   []


In [4]:
EXPORT = "2026-09-23"  # pin the export this notebook was written against

sentences_df = loading.load(DATA, "all_sentences.csv", EXPORT)
metrics_df = loading.load(DATA, "all_metrics.csv", EXPORT)
utterances_df = loading.load(DATA, "all_utterances.csv", EXPORT)

loaded all_sentences.csv  9519 rows
loaded all_metrics.csv  510 rows
loaded all_utterances.csv  1201 rows


In [5]:
#----------------------
# additional setup
#----------------------
import re

from IPython.display import display, Markdown


In [6]:
print("Original datasets: Sentences", sentences_df.shape, "| Utterances:", utterances_df.shape, "| Metrics:", metrics_df.shape, "\n")

print("SENTENCES", "\n")
print(f"Sentences dataset: {sentences_df.shape[0]:,} rows × {sentences_df.shape[1]} columns")
print("Sample Sentences dataset (top 3 rows):")
display(sentences_df.head(3))
print("\n", "\n")

print("UTTERANCES", "\n")
print(f"Utterances dataset: {utterances_df.shape[0]:,} rows × {utterances_df.shape[1]} columns")
print("Sample Utterances dataset (top 3 rows):")
display(utterances_df.head(3))
print("\n", "\n")

print("METRICS", "\n")
print(f"Metrics dataset: {metrics_df.shape[0]:,} rows × {metrics_df.shape[1]} columns")
print("Sample Metrics dataset (top 3 rows):")
display(metrics_df.head(3))

Original datasets: Sentences (9519, 12) | Utterances: (1201, 11) | Metrics: (510, 6) 

SENTENCES 

Sentences dataset: 9,519 rows × 12 columns
Sample Sentences dataset (top 3 rows):


,bank,quarter,call_type,call_date,position_in_call,section,speaker_role,speaker_name,speaker_institution,sentence_id,sentence_number,sentence
0,JPM,2023-Q1,earnings,2023-04-14,1,prepared,operator,Operator,NaN,19308,1,"Good morning, ladies and gentlemen."
1,JPM,2023-Q1,earnings,2023-04-14,1,prepared,operator,Operator,NaN,19309,2,Welcome to JPMorgan Chase’s First Quarter 2023...
2,JPM,2023-Q1,earnings,2023-04-14,1,prepared,operator,Operator,NaN,19310,3,This call is being recorded.



 

UTTERANCES 

Utterances dataset: 1,201 rows × 11 columns
Sample Utterances dataset (top 3 rows):


,bank,quarter,call_type,call_date,position_in_call,section,speaker_role,speaker_name,speaker_institution,text_length,text
0,JPM,2023-Q1,earnings,2023-04-14,1,prepared,operator,Operator,NaN,411,"Good morning, ladies and gentlemen. Welcome to..."
1,JPM,2023-Q1,earnings,2023-04-14,2,prepared,management,Jeremy Barnum,NaN,12559,"Thanks, and good morning, everyone. The presen..."
2,JPM,2023-Q1,earnings,2023-04-14,3,qa,operator,Operator,NaN,106,Please stand by. The first question is coming ...



 

METRICS 

Metrics dataset: 510 rows × 6 columns
Sample Metrics dataset (top 3 rows):


,bank,quarter,metric_name,value,unit,basis
0,JPM,2023-Q1,average_shares_basic,2968.50,count m,reported
1,JPM,2023-Q1,average_shares_diluted,2972.70,count m,reported
2,JPM,2023-Q1,basic_eps,4.11,USD,reported


In [7]:
# check the first and the last date of UBS earnings call in utterances_df
display(Markdown("**UBS:**"))
print("First earnings call in the database: ", utterances_df["call_date"].min())
print("Last earnings call in the database:", utterances_df["call_date"].max())

# check the first and the last date of JPM earnings call in utterances_df
display(Markdown("**JPM:**"))
print("First earnings call in the database: ", utterances_df["call_date"].min())
print("Last earnings call in the database:", utterances_df["call_date"].max())

**UBS:**

First earnings call in the database:  2023-04-14
Last earnings call in the database: 2025-02-04


**JPM:**

First earnings call in the database:  2023-04-14
Last earnings call in the database: 2025-02-04


# **Define the transaction-relative quarter labels.**

## Transaction-relative timeline

### Purpose

The UBS–Credit Suisse and JPMorgan–First Republic transactions occurred on different dates and followed different legal and operational timelines. Comparing calls only by calendar quarter could therefore obscure the stage of the acquisition being discussed.

To support a meaningful comparison, each earnings call is assigned a **transaction-relative quarter** and a **transaction phase**. This converts calendar time into event time, allowing the analysis to compare equivalent stages of the two acquisitions.

### Why the earnings-call date is used

The classification is based on the date on which the earnings call took place rather than solely on its reported financial quarter.

This distinction is important because a call discussing a particular reporting quarter may occur before or after a major transaction event. For example, JPMorgan’s 2023-Q1 earnings call occurred before its acquisition of First Republic Bank, while UBS’s 2023-Q1 call occurred after the announcement of the Credit Suisse transaction.

Using the actual call date ensures that the model only has access to information that was publicly available at the time of the call. This reduces the risk of look-ahead bias and improves the validity of the longitudinal analysis.

### Transaction dates

The analysis uses the following key dates:

| Acquirer | Target | Announcement date | Completion date |
|---|---|---:|---:|
| UBS | Credit Suisse | 19 March 2023 | 12 June 2023 |
| JPMorgan Chase | First Republic Bank | 1 May 2023 | 1 May 2023 |

For JPMorgan–First Republic, the announcement and completion occurred on the same date. Consequently, this transaction does not have a separate announcement-to-completion phase in the earnings-call data.

### Relative-quarter labels

Each call is assigned a label based on its calendar-quarter distance from the transaction announcement:

- `A-2`: two calendar quarters before the announcement
- `A-1`: one calendar quarter before the announcement
- `A0`: the calendar quarter containing the announcement
- `A+1`: one calendar quarter after the announcement
- `A+2`: two calendar quarters after the announcement

These labels provide a standardised event-time axis for comparing the two banks. However, calls within the same calendar quarter can still fall on different sides of the announcement date. The more detailed transaction-phase classification therefore uses exact dates.

### Transaction phases

Each call is also assigned to one of the following phases:

1. **Pre-announcement baseline** – the call occurred before the transaction was publicly announced.
2. **Announced / pre-completion** – the transaction had been announced but had not yet legally completed.
3. **Initial consolidation** – the call occurred during the first six months after completion.
4. **Early integration** – the call occurred between six and twelve months after completion.
5. **Ongoing integration** – the call occurred between twelve and twenty-four months after completion.
6. **Later integration / normalisation** – the call occurred at least twenty-four months after completion.

A broader variable also groups calls into `Pre-transaction`, `Announcement to completion`, and `Post-completion`. This less granular classification may be more appropriate where the number of calls in individual phases is small.

### Application across the datasets

The transaction labels are first added to the sentence-level and utterance-level datasets, both of which contain an earnings-call date.

The metrics dataset does not contain a call date. Its transaction labels are therefore added by matching each bank-quarter observation to the corresponding call date in the utterance dataset. This ensures that the same timeline definition is used across the qualitative and quantitative analysis.

### Analytical use

The transaction-relative variables will support three forms of comparison:

- **Within-bank comparison:** how the issues discussed by a bank change from the pre-announcement period through successive integration stages.
- **Cross-bank comparison:** whether similar risks emerge at equivalent transaction stages for UBS and JPMorgan.
- **Analyst–management comparison:** whether analysts raise concerns before management discusses them, or continue to question an issue after management presents it as controlled or improving.

The transaction phases should not be interpreted as evidence that risk necessarily increased or decreased. They provide a consistent time structure within which changes in risk-related communication can be examined.

### Methodological limitations

The phase boundaries are analytical conventions rather than estimates of the banks’ actual internal integration stages. Integration programmes may progress at different speeds, and significant developments can occur between quarterly calls.

In addition, the two transactions were structurally different. UBS acquired Credit Suisse through a negotiated emergency transaction followed by a distinct legal-completion period, whereas JPMorgan acquired substantial assets and liabilities of First Republic following its closure by the Federal Deposit Insurance Corporation. Comparisons should therefore focus on recurring risk signals and communication patterns rather than assuming that the transactions are directly equivalent.

For these reasons, both the original call date and financial quarter are retained alongside the derived labels. This preserves traceability and allows alternative phase definitions to be tested as part of robustness analysis.

In [8]:
# ============================================================
# Transaction-relative quarter and phase labels
# ============================================================

# Key transaction dates.
# "announcement_date" marks when the transaction became public.
# "completion_date" marks legal completion / acquisition.
TRANSACTIONS = {
    "UBS": {
        "target": "Credit Suisse",
        "transaction_name": "UBS–Credit Suisse",
        "announcement_date": "2023-03-19",
        "completion_date": "2023-06-12",
    },
    "JPM": {
        "target": "First Republic Bank",
        "transaction_name": "JPMorgan–First Republic",
        # The FDIC transaction was announced and completed on the same day.
        "announcement_date": "2023-05-01",
        "completion_date": "2023-05-01",
    },
}

In [9]:
def _calendar_quarter_number(dates):
    """
    Convert dates into monotonically increasing calendar-quarter numbers.

    Example:
        2023-Q1 -> 2023 * 4 + 0
        2023-Q2 -> 2023 * 4 + 1
    """
    dates = pd.to_datetime(dates)
    return dates.dt.year * 4 + dates.dt.quarter - 1


def add_transaction_labels(df):
    """
    Add transaction-relative labels to a dataframe containing:
        - bank
        - call_date

    Labels are based on the date of the earnings call, not merely the
    reporting-quarter name.

    New fields:
        acquisition_target
        transaction_name
        transaction_announcement_date
        transaction_completion_date
        days_from_announcement
        days_from_completion
        quarters_from_announcement
        quarters_from_completion
        relative_quarter
        transaction_phase
        transaction_period
    """
    labelled = df.copy()

    labelled["call_date"] = pd.to_datetime(
        labelled["call_date"], errors="coerce"
    )

    # Map transaction metadata to every row.
    for field in [
        "target",
        "transaction_name",
        "announcement_date",
        "completion_date",
    ]:
        mapping = {
            bank: details[field]
            for bank, details in TRANSACTIONS.items()
        }

        output_name = {
            "target": "acquisition_target",
            "transaction_name": "transaction_name",
            "announcement_date": "transaction_announcement_date",
            "completion_date": "transaction_completion_date",
        }[field]

        labelled[output_name] = labelled["bank"].map(mapping)

    labelled["transaction_announcement_date"] = pd.to_datetime(
        labelled["transaction_announcement_date"], errors="coerce"
    )
    labelled["transaction_completion_date"] = pd.to_datetime(
        labelled["transaction_completion_date"], errors="coerce"
    )

    # Exact distance from the transaction dates.
    labelled["days_from_announcement"] = (
        labelled["call_date"]
        - labelled["transaction_announcement_date"]
    ).dt.days

    labelled["days_from_completion"] = (
        labelled["call_date"]
        - labelled["transaction_completion_date"]
    ).dt.days

    # Calendar-quarter distance from each event.
    call_q = _calendar_quarter_number(labelled["call_date"])
    announcement_q = _calendar_quarter_number(
        labelled["transaction_announcement_date"]
    )
    completion_q = _calendar_quarter_number(
        labelled["transaction_completion_date"]
    )

    labelled["quarters_from_announcement"] = (
        call_q - announcement_q
    ).astype("Int64")

    labelled["quarters_from_completion"] = (
        call_q - completion_q
    ).astype("Int64")

    # Compact label for charts and tables:
    # A-2 = two calendar quarters before announcement
    # A0  = same calendar quarter as announcement
    # A+1 = one calendar quarter after announcement
    def format_relative_quarter(value):
        if pd.isna(value):
            return pd.NA

        value = int(value)

        if value == 0:
            return "A0"
        if value > 0:
            return f"A+{value}"
        return f"A{value}"

    labelled["relative_quarter"] = (
        labelled["quarters_from_announcement"]
        .map(format_relative_quarter)
        .astype("string")
    )

    # Detailed transaction phase based on exact call dates.
    conditions = [
        labelled["call_date"] < labelled["transaction_announcement_date"],

        (
            labelled["call_date"]
            >= labelled["transaction_announcement_date"]
        )
        & (
            labelled["call_date"]
            < labelled["transaction_completion_date"]
        ),

        (
            labelled["call_date"]
            >= labelled["transaction_completion_date"]
        )
        & (
            labelled["days_from_completion"] < 180
        ),

        (
            labelled["days_from_completion"] >= 180
        )
        & (
            labelled["days_from_completion"] < 365
        ),

        (
            labelled["days_from_completion"] >= 365
        )
        & (
            labelled["days_from_completion"] < 730
        ),

        labelled["days_from_completion"] >= 730,
    ]

    phases = [
        "1. Pre-announcement baseline",
        "2. Announced / pre-completion",
        "3. Initial consolidation (0–6 months)",
        "4. Early integration (6–12 months)",
        "5. Ongoing integration (12–24 months)",
        "6. Later integration / normalisation (24+ months)",
    ]

    labelled["transaction_phase"] = np.select(
        conditions,
        phases,
        default="Unclassified",
    )

    # Broader label useful where the number of observations is small.
    labelled["transaction_period"] = np.select(
        [
            labelled["call_date"]
            < labelled["transaction_announcement_date"],

            (
                labelled["call_date"]
                >= labelled["transaction_announcement_date"]
            )
            & (
                labelled["call_date"]
                < labelled["transaction_completion_date"]
            ),

            labelled["call_date"]
            >= labelled["transaction_completion_date"],
        ],
        [
            "Pre-transaction",
            "Announcement to completion",
            "Post-completion",
        ],
        default="Unclassified",
    )

    return labelled


# Apply the labels to sentence- and utterance-level data.
sentences_df = add_transaction_labels(sentences_df)
utterances_df = add_transaction_labels(utterances_df)


# ============================================================
# Add the same labels to metrics_df
# ============================================================

# metrics_df has no call_date, so construct one bank-quarter lookup from
# the transcript datasets. This avoids inventing a date for each quarter.
call_dates = (
    utterances_df[
        [
            "bank",
            "quarter",
            "call_date",
            "acquisition_target",
            "transaction_name",
            "transaction_announcement_date",
            "transaction_completion_date",
            "days_from_announcement",
            "days_from_completion",
            "quarters_from_announcement",
            "quarters_from_completion",
            "relative_quarter",
            "transaction_phase",
            "transaction_period",
        ]
    ]
    .drop_duplicates(subset=["bank", "quarter"])
)

metrics_df = metrics_df.merge(
    call_dates,
    on=["bank", "quarter"],
    how="left",
    validate="many_to_one",
)


# Make the phase order explicit for tables and charts.
PHASE_ORDER = [
    "1. Pre-announcement baseline",
    "2. Announced / pre-completion",
    "3. Initial consolidation (0–6 months)",
    "4. Early integration (6–12 months)",
    "5. Ongoing integration (12–24 months)",
    "6. Later integration / normalisation (24+ months)",
    "Unclassified",
]

for df in [sentences_df, utterances_df, metrics_df]:
    df["transaction_phase"] = pd.Categorical(
        df["transaction_phase"],
        categories=PHASE_ORDER,
        ordered=True,
    )


# ============================================================
# Validate the resulting bank-quarter timeline
# ============================================================

# timeline = (
#     utterances_df[
#         [
#             "bank",
#             "quarter",
#             "call_date",
#             "transaction_name",
#             "relative_quarter",
#             "days_from_announcement",
#             "days_from_completion",
#             "transaction_phase",
#             "transaction_period",
#         ]
#     ]
#     .drop_duplicates(subset=["bank", "quarter"])
#     .sort_values(["bank", "call_date"])
#     .reset_index(drop=True)
# )

# display(timeline)

# **Create a high-recall set of M&A-related utterances**

#### Use utterances as the main analytical unit

An utterance preserves a speaker's full contribution, which is important for understanding qualifications, uncertainty and management responses. Sentence-level records can later provide more precise citations.

In [10]:
# Remove operator statements because they contain
# procedural language rather than substantive risk information
analysis_utterances = (
    utterances_df[
        utterances_df["speaker_role"].ne("operator")
    ]
    .copy()
    .reset_index(drop=True)
)

print("Utterances available for analysis:", len(analysis_utterances))

Utterances available for analysis: 1083


#### Create a transparent first-stage keyword filter

Start with broad, bank-specific transaction terms and general integration-risk terms.

In [11]:
TRANSACTION_TERMS = {
    "UBS": [
        "credit suisse",
        "cs",
    ],
    "JPM": [
        "first republic bank",
        "first republic",
        "frb",
    ],
}

GENERAL_MA_TERMS = [
    "acquisition",
    "acquired",
    "transaction",
    "merger",
    "integration",
    "integrate",
    "integrating",
    "migration",
    "conversion",
    "restructuring",
    "synergy",
    "synergies",
    "run-off",
    "wind-down",
    "non-core",
    "client attrition",
    "client retention",
    "deposit attrition",
    "deposit retention",
    "integration cost",
    "integration costs",
    "restructuring cost",
    "restructuring costs",
    "legal close",
    "closing",
]

In [12]:
def contains_ma_terms(row):
    text = str(row["text"]).lower()

    terms = (
        TRANSACTION_TERMS.get(row["bank"], [])
        + GENERAL_MA_TERMS
    )

    return any(
        re.search(rf"\b{re.escape(term)}\b", text)
        for term in terms
    )


analysis_utterances["keyword_ma_relevant"] = (
    analysis_utterances.apply(contains_ma_terms, axis=1)
)

ma_utterances = (
    analysis_utterances[
        analysis_utterances["keyword_ma_relevant"]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Retrieved {len(ma_utterances):,} potentially M&A-related "
    f"utterances from {len(analysis_utterances):,} substantive utterances."
)

Retrieved 222 potentially M&A-related utterances from 1,083 substantive utterances.


#### Review retrieval coverage

In [13]:
# Examine how many utterances were selected by bank, quarter, phase and speaker role
retrieval_summary = (
    analysis_utterances
    .groupby(
        [
            "bank",
            "quarter",
            "transaction_phase",
            "speaker_role",
        ],
        observed=True,
    )
    .agg(
        total_utterances=("text", "size"),
        retrieved_utterances=("keyword_ma_relevant", "sum"),
    )
    .reset_index()
)

retrieval_summary["retrieval_share"] = (
    retrieval_summary["retrieved_utterances"]
    / retrieval_summary["total_utterances"]
)

display(retrieval_summary)

,bank,quarter,transaction_phase,speaker_role,total_utterances,retrieved_utterances,retrieval_share
0,JPM,2023-Q1,1. Pre-announcement baseline,analyst,43,0,0.000000
1,JPM,2023-Q1,1. Pre-announcement baseline,management,55,2,0.036364
2,JPM,2023-Q2,3. Initial consolidation (0–6 months),analyst,81,11,0.135802
3,JPM,2023-Q2,3. Initial consolidation (0–6 months),management,84,18,0.214286
4,JPM,2023-Q3,3. Initial consolidation (0–6 months),analyst,36,5,0.138889
5,JPM,2023-Q3,3. Initial consolidation (0–6 months),management,53,8,0.150943
6,JPM,2023-Q4,4. Early integration (6–12 months),analyst,31,4,0.129032
7,JPM,2023-Q4,4. Early integration (6–12 months),management,28,5,0.178571
8,JPM,2024-Q1,4. Early integration (6–12 months),analyst,34,3,0.088235
9,JPM,2024-Q1,4. Early integration (6–12 months),management,38,5,0.131579


#### Inspect a sample of selected passages

In [14]:
display(
    ma_utterances[
        [
            "bank",
            "quarter",
            "transaction_phase",
            "section",
            "speaker_role",
            "speaker_name",
            "text",
        ]
    ].sample(
        min(20, len(ma_utterances)),
        random_state=42,
    )
)

,bank,quarter,transaction_phase,section,speaker_role,speaker_name,text
154,UBS,2023-Q4,4. Early integration (6–12 months),qa,analyst,Stefan Stalmann,"Yes. Good morning, everyone. Thanks for the pr..."
215,UBS,2024-Q4,5. Ongoing integration (12–24 months),qa,analyst,Kian Abouhossein,Thank you. Okay. And just on the 6 billion in ...
15,JPM,2023-Q2,3. Initial consolidation (0–6 months),qa,management,Jamie Dimon,"Yeah. So, the first one, now – look, I was jus..."
173,UBS,2024-Q1,4. Early integration (6–12 months),qa,analyst,Jeremy Sigee,"Thank you. Good morning. Two questions, please..."
158,UBS,2023-Q4,4. Early integration (6–12 months),qa,analyst,Benjamin Goy,"Yes. Hi. Good morning. Two questions, please, ..."
125,UBS,2023-Q3,3. Initial consolidation (0–6 months),qa,management,Sergio P. Ermotti,Okay. Thank you. So in respect of the capital ...
97,UBS,2023-Q2,3. Initial consolidation (0–6 months),qa,analyst,Jeremy Sigee,Good morning. Thank you very much for all the ...
25,JPM,2023-Q2,3. Initial consolidation (0–6 months),qa,management,Jeremy Barnum,"Yeah. Good question, Steve, so let me say a co..."
60,JPM,2024-Q1,4. Early integration (6–12 months),qa,management,Jeremy Barnum,"Okay. Thanks, Charlie. And I'm going to do my ..."
68,JPM,2024-Q3,5. Ongoing integration (12–24 months),qa,management,Jeremy Barnum,"I would say, Matt, generally no, frankly, with..."


#### Inspect excluded passages

In [15]:
excluded_sample = (
    analysis_utterances[
        ~analysis_utterances["keyword_ma_relevant"]
    ]
    .sample(
        min(
            20,
            (~analysis_utterances["keyword_ma_relevant"]).sum(),
        ),
        random_state=42,
    )
)

display(
    excluded_sample[
        [
            "bank",
            "quarter",
            "speaker_role",
            "speaker_name",
            "text",
        ]
    ]
)

,bank,quarter,speaker_role,speaker_name,text
868,UBS,2023-Q4,analyst,Giulia Miotto,Thanks.
677,JPM,2024-Q4,management,Jamie Dimon,I would just point the biggest driver of credi...
139,JPM,2023-Q2,management,Jamie Dimon,We're not. And I said – I think they did do a ...
234,JPM,2023-Q2,analyst,Betsy L. Graseck,So then…
438,JPM,2024-Q1,analyst,Ebrahim H. Poonawala,Helpful. Thank you so much.
1021,UBS,2024-Q3,management,Todd Tuckner,"Yeah. Hi, Piers. So on the second look, you kn..."
646,JPM,2024-Q4,analyst,Mike Mayo,"So, you'll stay around maybe for a few more ye..."
301,JPM,2023-Q3,analyst,Gerard Cassidy,"Good morning, Jeremy. How are you?"
627,JPM,2024-Q3,management,Jeremy Barnum,"Thanks, Saul. Thanks, everyone."
1039,UBS,2024-Q4,management,Sergio P. Ermotti,"Thank you, Kian. I'm sorry, and I really appre..."


#### Preserve Q&A context

A selected analyst question should be paired with the following management response. Keyword filtering each utterance independently may retrieve the question but miss the answer because management might respond with phrases such as “that” or “the issue” without repeating the acquisition's name.

To preserve conversational context, analyst questions were linked to their associated management responses to form Q&A exchanges. Where a reliable question-answer boundary could not be identified, neighbouring utterances were retained as a context window.

In [16]:
# print values in the speaker_role column
print("Speaker roles in the utterances df: ", utterances_df['speaker_role'].unique())
print("Speaker roles it the sentences df: ", sentences_df['speaker_role'].unique())

Speaker roles in the utterances df:  ['operator' 'management' 'analyst' 'ir']
Speaker roles it the sentences df:  ['operator' 'management' 'analyst' 'ir']


In [17]:
# ------------------------------------------------------------
# 1. Define retrieval terms
# ------------------------------------------------------------

def build_regex(terms):
    """
    Construct one case-insensitive regex from a list of terms.

    Longer phrases are placed first so that specific expressions are
    tested before their shorter components.
    """
    unique_terms = sorted(
        set(terms),
        key=len,
        reverse=True,
    )

    return "|".join(
        rf"\b{re.escape(term)}\b"
        for term in unique_terms
    )


general_ma_pattern = build_regex(GENERAL_MA_TERMS)

bank_ma_patterns = {
    bank: build_regex(terms)
    for bank, terms in TRANSACTION_TERMS.items()
}


# ------------------------------------------------------------
# 2. Select and order Q&A utterances
# ------------------------------------------------------------

# Keep the original dataframe unchanged.
qa_turns = (
    utterances_df.loc[
        utterances_df["section"].astype(str).str.lower().eq("qa")
        & utterances_df["speaker_role"].isin(
            ["analyst", "management"]
        )
    ]
    .copy()
    .sort_values(
        [
            "bank",
            "quarter",
            "call_date",
            "position_in_call",
        ]
    )
    .reset_index(drop=True)
)

# Operator and investor-relations utterances are excluded because they
# are generally procedural rather than part of the substantive exchange.
print(f"Substantive Q&A utterances: {len(qa_turns):,}")

Substantive Q&A utterances: 1,048


In [18]:
# ------------------------------------------------------------
# 3. Identify the start of each Q&A exchange
# ------------------------------------------------------------

call_keys = ["bank", "quarter", "call_date"]

is_analyst = qa_turns["speaker_role"].eq("analyst")

# On each row, record the most recently observed analyst in that call.
current_analyst = (
    qa_turns["speaker_name"]
    .where(is_analyst)
    .groupby(
        [qa_turns[column] for column in call_keys],
        sort=False,
    )
    .ffill()
)

# Shift within each call to obtain the analyst attached to the
# preceding substantive turn.
previous_analyst = (
    current_analyst
    .groupby(
        [qa_turns[column] for column in call_keys],
        sort=False,
    )
    .shift()
)

# A new exchange begins when:
#   1. the current speaker is an analyst; and
#   2. the preceding analyst in the call is absent or different.
starts_new_exchange = (
    is_analyst
    & (
        previous_analyst.isna()
        | qa_turns["speaker_name"].ne(previous_analyst)
    )
)

# Number exchanges independently within each earnings call.
qa_turns["exchange_number"] = (
    starts_new_exchange
    .groupby(
        [qa_turns[column] for column in call_keys],
        sort=False,
    )
    .cumsum()
)

# Remove management remarks occurring before the first analyst question.
qa_turns = (
    qa_turns.loc[qa_turns["exchange_number"].gt(0)]
    .copy()
)

# Construct a stable, human-readable exchange identifier.
qa_turns["exchange_id"] = (
    qa_turns["bank"].astype(str)
    + "_"
    + qa_turns["quarter"].astype(str)
    + "_QA"
    + qa_turns["exchange_number"]
      .astype(int)
      .astype(str)
      .str.zfill(2)
)

# Number each turn within its exchange.
qa_turns["turn_in_exchange"] = (
    qa_turns.groupby(
        call_keys + ["exchange_number"],
        sort=False,
    )
    .cumcount()
    .add(1)
)

print(
    "Q&A exchanges constructed:",
    f"{qa_turns['exchange_id'].nunique():,}",
)

Q&A exchanges constructed: 172


In [19]:
# ------------------------------------------------------------
# 4. Identify potentially M&A-related utterances
# ------------------------------------------------------------

text = qa_turns["text"].fillna("").astype(str)

# General M&A terminology applies to both banks.
general_match = text.str.contains(
    general_ma_pattern,
    case=False,
    regex=True,
    na=False,
)

# Begin with the general-term result.
qa_turns["utterance_ma_relevant"] = general_match

# Add transaction-specific matches for each bank.
for bank, pattern in bank_ma_patterns.items():
    bank_match = (
        qa_turns["bank"].eq(bank)
        & text.str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
    )

    qa_turns["utterance_ma_relevant"] |= bank_match


# Retain the entire exchange when at least one constituent utterance
# matches the retrieval terms.
qa_turns["exchange_ma_relevant"] = (
    qa_turns
    .groupby("exchange_id", sort=False)[
        "utterance_ma_relevant"
    ]
    .transform("any")
)

ma_qa_turns = (
    qa_turns.loc[qa_turns["exchange_ma_relevant"]]
    .copy()
    .reset_index(drop=True)
)

print(
    "M&A-related exchanges:",
    f"{ma_qa_turns['exchange_id'].nunique():,}",
)

print(
    "Directly matching utterances:",
    f"{ma_qa_turns['utterance_ma_relevant'].sum():,}",
)

print(
    "Total utterances retained with context:",
    f"{len(ma_qa_turns):,}",
)

print(
    "Additional contextual utterances:",
    f"{len(ma_qa_turns) - ma_qa_turns['utterance_ma_relevant'].sum():,}",
)

M&A-related exchanges: 102
Directly matching utterances: 192
Total utterances retained with context: 590
Additional contextual utterances: 398


In [22]:
# ------------------------------------------------------------
# 5. Format each speaker turn
# ------------------------------------------------------------

ma_qa_turns["formatted_turn"] = (
    "["
    + ma_qa_turns["speaker_role"].str.upper()
    + " — "
    + ma_qa_turns["speaker_name"].fillna("Unknown speaker")
    + "]\n"
    + ma_qa_turns["text"].fillna("")
)

# Create role-specific text columns before aggregation.
ma_qa_turns["analyst_text"] = (
    ma_qa_turns["text"]
    .where(ma_qa_turns["speaker_role"].eq("analyst"))
)

ma_qa_turns["management_text"] = (
    ma_qa_turns["text"]
    .where(ma_qa_turns["speaker_role"].eq("management"))
)

ma_qa_turns["analyst_name"] = (
    ma_qa_turns["speaker_name"]
    .where(ma_qa_turns["speaker_role"].eq("analyst"))
)

ma_qa_turns["management_name"] = (
    ma_qa_turns["speaker_name"]
    .where(ma_qa_turns["speaker_role"].eq("management"))
)

In [23]:
#--------------------
# helper aggregations
#--------------------

def combine_text(series):
    """Combine non-missing text in its original order."""
    return "\n\n".join(
        series.dropna().astype(str).str.strip()
    )


def combine_unique(series):
    """Combine unique, non-missing values in their original order."""
    return "; ".join(
        pd.unique(
            series.dropna().astype(str).str.strip()
        )
    )

In [24]:
# ------------------------------------------------------------
# 6. Produce the exchange-level analytical dataset
# ------------------------------------------------------------

metadata_aggregations = {
    "bank": ("bank", "first"),
    "quarter": ("quarter", "first"),
    "call_date": ("call_date", "first"),
    "section": ("section", "first"),
    "analyst_names": ("analyst_name", combine_unique),
    "management_names": ("management_name", combine_unique),
    "analyst_question": ("analyst_text", combine_text),
    "management_response": ("management_text", combine_text),
    "llm_input_text": ("formatted_turn", combine_text),
    "first_position": ("position_in_call", "min"),
    "last_position": ("position_in_call", "max"),
    "number_of_turns": ("text", "size"),
    "keyword_matching_turns": (
        "utterance_ma_relevant",
        "sum",
    ),
}

# Include transaction labels if they are available.
optional_columns = {
    "transaction_name": ("transaction_name", "first"),
    "transaction_phase": ("transaction_phase", "first"),
    "transaction_period": ("transaction_period", "first"),
    "relative_quarter": ("relative_quarter", "first"),
    "days_from_announcement": (
        "days_from_announcement",
        "first",
    ),
    "days_from_completion": (
        "days_from_completion",
        "first",
    ),
}

for output_column, aggregation in optional_columns.items():
    source_column = aggregation[0]

    if source_column in ma_qa_turns.columns:
        metadata_aggregations[output_column] = aggregation


ma_qa_exchanges = (
    ma_qa_turns
    .groupby("exchange_id", as_index=False, sort=False)
    .agg(**metadata_aggregations)
    .sort_values(
        ["bank", "call_date", "first_position"]
    )
    .reset_index(drop=True)
)

print(
    "Final exchange-level dataset:",
    ma_qa_exchanges.shape,
)

display(
    ma_qa_exchanges[
        [
            "exchange_id",
            "bank",
            "quarter",
            "transaction_phase",
            "analyst_question",
            "management_response",
        ]
    ].head()
)

Final exchange-level dataset: (102, 20)


,exchange_id,bank,quarter,transaction_phase,analyst_question,management_response
0,JPM_2023-Q1_QA04,JPM,2023-Q1,1. Pre-announcement baseline,Hi. Good morning. My first question is you men...,"Yeah. So, Erika, as you know, we take – not go..."
1,JPM_2023-Q2_QA01,JPM,2023-Q2,3. Initial consolidation (0–6 months),Hey. Good morning.\n\nSo congrats on the deal....,"Good morning.\n\nYeah. Good question, Steve. I..."
2,JPM_2023-Q2_QA02,JPM,2023-Q2,3. Initial consolidation (0–6 months),"Yes, thank you. Jeremy, my first question is h...","Okay. A lot of technical details in there, Eri..."
3,JPM_2023-Q2_QA03,JPM,2023-Q2,3. Initial consolidation (0–6 months),"Thank you. Good morning, Jeremy.\n\nJeremy and...","Good morning, Gerard.\n\nWe're leaving behind ..."
4,JPM_2023-Q2_QA04,JPM,2023-Q2,3. Initial consolidation (0–6 months),"Good morning. Jeremy, if I could follow up jus...","Yeah. Good question, Ebrahim. I guess, in simp..."


In [25]:
# ------------------------------------------------------------
# 7. Automated validation checks
# ------------------------------------------------------------

# Every exchange should contain at least one analyst turn.
assert (
    ma_qa_turns
    .groupby("exchange_id")["speaker_role"]
    .apply(lambda roles: roles.eq("analyst").any())
    .all()
), "At least one exchange has no analyst question."

# Every retained exchange should contain a keyword-matching utterance.
assert (
    ma_qa_turns
    .groupby("exchange_id")["utterance_ma_relevant"]
    .any()
    .all()
), "At least one retained exchange has no M&A keyword match."

# Exchange IDs should uniquely identify exchange-level rows.
assert ma_qa_exchanges["exchange_id"].is_unique

# Display exchanges without a management response for manual review.
unanswered_exchanges = ma_qa_exchanges.loc[
    ma_qa_exchanges["management_response"].eq("")
]

print(
    "Exchanges without an identified management response:",
    len(unanswered_exchanges),
)

display(
    unanswered_exchanges[
        [
            "exchange_id",
            "bank",
            "quarter",
            "analyst_question",
        ]
    ].head()
)

Exchanges without an identified management response: 0


,exchange_id,bank,quarter,analyst_question


In [26]:
# review a sample
sample_size = min(10, len(ma_qa_exchanges))

sample_exchanges = ma_qa_exchanges.sample(
    n=sample_size,
    random_state=42,
)

for row in sample_exchanges.itertuples(index=False):
    print("=" * 80)
    print(row.exchange_id)
    print("=" * 80)
    print(row.llm_input_text)
    print()

JPM_2024-Q4_QA06
[ANALYST — Betsy L. Graseck]
Hi. Good morning.

[MANAGEMENT — Jeremy Barnum]
Hey, Betsy.

[MANAGEMENT — Jamie Dimon]
Hey, Betsy.

[ANALYST — Betsy L. Graseck]
Guys, can you hear me okay? All right. Just want to make sure you can hear me. So this has been a great call. First, congratulations on a great quarter. It's been a great call with a lot of robust questions here. You're like, what else is there to ask? Here's my question. As we think about the NII outlook and you highlighted the NIM pressure, but loan – but basically, balance is increasing here, right? Maybe you could just speak to – could you help me understand the order of the drivers? Is it QT going away, deposits going up, punch it into securities? Is it loan growth inflecting? Is there any place in the franchise where you see loan growth opportunities for inflection this coming year? And then lastly, as I think about your comments around you've got the green market share number one, you've got the yellow, yo

In [27]:
# analyse retrieval coverage
retrieval_coverage = (
    qa_turns
    .groupby(
        [
            "bank",
            "quarter",
            "transaction_phase",
        ],
        observed=True,
    )
    .agg(
        total_exchanges=("exchange_id", "nunique"),
        ma_exchanges=(
            "exchange_ma_relevant",
            lambda values: qa_turns.loc[
                values.index[values],
                "exchange_id",
            ].nunique()
        ),
    )
    .reset_index()
)

retrieval_coverage["ma_exchange_share"] = (
    retrieval_coverage["ma_exchanges"]
    / retrieval_coverage["total_exchanges"]
)

display(retrieval_coverage)

,bank,quarter,transaction_phase,total_exchanges,ma_exchanges,ma_exchange_share
0,JPM,2023-Q1,1. Pre-announcement baseline,11,1,0.090909
1,JPM,2023-Q2,3. Initial consolidation (0–6 months),12,10,0.833333
2,JPM,2023-Q3,3. Initial consolidation (0–6 months),10,6,0.600000
3,JPM,2023-Q4,4. Early integration (6–12 months),10,4,0.400000
4,JPM,2024-Q1,4. Early integration (6–12 months),11,3,0.272727
5,JPM,2024-Q2,5. Ongoing integration (12–24 months),10,4,0.400000
6,JPM,2024-Q3,5. Ongoing integration (12–24 months),10,2,0.200000
7,JPM,2024-Q4,5. Ongoing integration (12–24 months),8,1,0.125000
8,UBS,2023-Q1,2. Announced / pre-completion,13,12,0.923077
9,UBS,2023-Q2,3. Initial consolidation (0–6 months),13,12,0.923077


In [28]:
# Retrieve relevant prepared remarks separately

prepared_remarks = (
    utterances_df.loc[
        utterances_df["section"].astype(str).str.lower().eq("prepared")
        & utterances_df["speaker_role"].eq("management")
    ]
    .copy()
    .sort_values(
        [
            "bank",
            "quarter",
            "call_date",
            "position_in_call",
        ]
    )
    .reset_index(drop=True)
)

prepared_text = prepared_remarks["text"].fillna("").astype(str)

prepared_remarks["ma_relevant"] = prepared_text.str.contains(
    general_ma_pattern,
    case=False,
    regex=True,
    na=False,
)

for bank, pattern in bank_ma_patterns.items():
    prepared_remarks["ma_relevant"] |= (
        prepared_remarks["bank"].eq(bank)
        & prepared_text.str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
    )

ma_prepared_remarks = (
    prepared_remarks.loc[prepared_remarks["ma_relevant"]]
    .copy()
    .reset_index(drop=True)
)

print(
    "M&A-related prepared remarks:",
    len(ma_prepared_remarks),
)

M&A-related prepared remarks: 30


We now have two distinct analytical sources:
- `ma_qa_exchanges`: analyst concerns and management responses
- `ma_prepared_remarks`: management's proactive narrative

In [1]:
# save the produced datasets
from pathlib import Path

CHECKPOINT_DIR = DATA / "llm_analysis_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

datasets_to_save = {
    "ma_qa_exchanges": ma_qa_exchanges,
    "ma_qa_turns": ma_qa_turns,
    "ma_prepared_remarks": ma_prepared_remarks,
    "retrieval_coverage": retrieval_coverage,
}

for name, dataframe in datasets_to_save.items():
    path = CHECKPOINT_DIR / f"{name}.pkl"
    dataframe.to_pickle(path)
    print(f"Saved {name}: {len(dataframe):,} rows")

NameError: name 'DATA' is not defined

### **SWITCH TO GPU**

Hosted runtime type: A100.

In [2]:
# verify the GPU
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [3]:
# reconnect Google Drive and reload the results
from google.colab import drive
from pathlib import Path
import pandas as pd

# Connect Google Drive
drive.mount("/content/drive", force_remount=False)

DATA = Path("/content/drive/MyDrive/boe-data")
CHECKPOINT_DIR = DATA / "llm_analysis_checkpoints"

# Verify that the checkpoint directory exists
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(
        f"Checkpoint directory not found: {CHECKPOINT_DIR}"
    )

# Reload the CPU-stage datasets
ma_qa_exchanges = pd.read_pickle(
    CHECKPOINT_DIR / "ma_qa_exchanges.pkl"
)

ma_qa_turns = pd.read_pickle(
    CHECKPOINT_DIR / "ma_qa_turns.pkl"
)

ma_prepared_remarks = pd.read_pickle(
    CHECKPOINT_DIR / "ma_prepared_remarks.pkl"
)

retrieval_coverage = pd.read_pickle(
    CHECKPOINT_DIR / "retrieval_coverage.pkl"
)

print("Datasets restored successfully:")
print("ma_qa_exchanges:", ma_qa_exchanges.shape)
print("ma_qa_turns:", ma_qa_turns.shape)
print("ma_prepared_remarks:", ma_prepared_remarks.shape)
print("retrieval_coverage:", retrieval_coverage.shape)

Mounted at /content/drive


FileNotFoundError: Checkpoint directory not found: /content/drive/MyDrive/boe-data/llm_analysis_checkpoints

In [3]:
# install the GPU-model packages
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 59.4 MB/s eta 0:00:00


# Run an LLM relevance-classification stage

In [4]:
# ============================================================
# Load a free, open-weight instruction-tuned LLM
# ============================================================

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Select a GPU under "
        "Runtime > Change runtime type, then rerun this cell."
    )

print("GPU:", torch.cuda.get_device_name(0))

# Load the model in 4-bit precision to reduce GPU-memory use.
quantisation_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=False,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=False,
    quantization_config=quantisation_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.eval()

print("Model loaded:", MODEL_NAME)

GPU: NVIDIA A100-SXM4-80GB


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-4B-Instruct-2507


In [5]:
# Define the supervisory risk taxonomy

RISK_TAXONOMY = {
    "capital_liquidity": (
        "Capital adequacy, liquidity, funding, deposits or "
        "balance-sheet capacity."
    ),
    "credit_asset_quality": (
        "Credit losses, loan quality, valuation or acquired assets."
    ),
    "integration_execution": (
        "Integration timetable, delivery risk, migration or execution."
    ),
    "operational_resilience": (
        "Technology, systems, data, cyber risk or operational continuity."
    ),
    "governance_risk_management": (
        "Governance, controls, accountability or risk-management capability."
    ),
    "legal_regulatory_conduct": (
        "Litigation, regulatory requirements, conduct or compliance."
    ),
    "costs_synergies_profitability": (
        "Integration costs, restructuring, synergies, revenue or profitability."
    ),
    "client_franchise": (
        "Client retention, deposit attrition, reputation or franchise strength."
    ),
    "workforce_culture": (
        "Staff retention, redundancies, incentives or cultural integration."
    ),
    "complexity_interconnectedness": (
        "Organisational complexity, systemic importance or interconnectedness."
    ),
    "other": (
        "Material acquisition-related risk that does not fit another category."
    ),
}

In [6]:
# Create relevance prompt

import json


SYSTEM_PROMPT = """
You are analysing bank earnings-call transcripts for prudential
supervisory research.

Your task is to classify whether a Q&A exchange is materially related
to the specified acquisition.

Rules:
1. Use only the supplied transcript text.
2. Do not use external knowledge.
3. Do not infer facts or risks that are not supported by the text.
4. An exchange is relevant only if it substantively discusses the
   acquisition, acquired institution, integration process, acquired
   assets or liabilities, transaction effects, or resulting risks.
5. A passing reference with no substantive discussion is not sufficient.
6. If the evidence is ambiguous, classify the exchange as "uncertain".
7. Return valid JSON only. Do not include Markdown or commentary.
"""


def format_taxonomy(taxonomy):
    return "\n".join(
        f"- {name}: {description}"
        for name, description in taxonomy.items()
    )


def build_relevance_prompt(row):
    return f"""
Classify the following earnings-call Q&A exchange.

Exchange ID: {row['exchange_id']}
Bank: {row['bank']}
Reporting quarter: {row['quarter']}
Transaction: {row.get('transaction_name', 'Not supplied')}
Transaction phase: {row.get('transaction_phase', 'Not supplied')}

Permitted risk categories:
{format_taxonomy(RISK_TAXONOMY)}

Transcript:
--- START OF TRANSCRIPT ---
{row['llm_input_text']}
--- END OF TRANSCRIPT ---

Return exactly one JSON object using this structure:

{{
  "exchange_id": "{row['exchange_id']}",
  "ma_relevance": "relevant, not_relevant, or uncertain",
  "reason": "Brief explanation grounded in the transcript",
  "risk_categories": ["zero or more permitted category names"],
  "evidence_turns": [1],
  "confidence": "high, medium, or low"
}}

The evidence_turns values must refer to the order of the transcript turns.
Use an empty risk_categories list and evidence_turns list if the exchange
is not relevant.
""".strip()

In [7]:
# Define the model-call function
def generate_response(
    user_prompt,
    max_new_tokens=400,
):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    # return_dict=True produces a BatchEncoding containing
    # input_ids and attention_mask.
    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    # Move every input tensor to the model's device.
    model_inputs = model_inputs.to(model.device)

    # Record the input length so that it can be removed from the output.
    input_length = model_inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens.
    generated_ids = output_ids[0, input_length:]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return response.strip()

In [8]:
# test one exchange
test_row = ma_qa_exchanges.iloc[0]

test_prompt = build_relevance_prompt(test_row)
test_output = generate_response(test_prompt)

print(test_output)

{
  "exchange_id": "JPM_2023-Q1_QA04",
  "ma_relevance": "not_relevant",
  "reason": "The exchange discusses reserve assumptions, unemployment rates, and ROTCE under recession scenarios, but does not mention the JPMorgan–First Republic acquisition, integration, assets, liabilities, costs, or any acquisition-related risks or execution details. There is no substantive discussion of the transaction phase or its implications.",
  "risk_categories": [],
  "evidence_turns": [],
  "confidence": "high"
}


In [9]:
# parse the result
def parse_json_response(response_text):
    """
    Parse the model response, removing Markdown fences if the model
    returns them despite the prompt.
    """
    cleaned = response_text.strip()

    if cleaned.startswith("```"):
        cleaned = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            cleaned,
            flags=re.IGNORECASE,
        ).strip()

    return json.loads(cleaned)


test_result = parse_json_response(test_output)
test_result

{'exchange_id': 'JPM_2023-Q1_QA04',
 'ma_relevance': 'not_relevant',
 'reason': 'The exchange discusses reserve assumptions, unemployment rates, and ROTCE under recession scenarios, but does not mention the JPMorgan–First Republic acquisition, integration, assets, liabilities, costs, or any acquisition-related risks or execution details. There is no substantive discussion of the transaction phase or its implications.',
 'risk_categories': [],
 'evidence_turns': [],
 'confidence': 'high'}

In [11]:
# run a small pilot
pilot_sample = (
    ma_qa_exchanges
    .groupby("bank", group_keys=False)
    .sample(
        n=min(
            5,
            ma_qa_exchanges.groupby("bank").size().min(),
        ),
        random_state=42,
    )
    .reset_index(drop=True)
)

pilot_results = []

for row in pilot_sample.to_dict("records"):
    prompt = build_relevance_prompt(row)
    raw_output = generate_response(prompt)

    try:
        parsed_output = parse_json_response(raw_output)
        parsed_output["parse_success"] = True
        parsed_output["raw_output"] = raw_output

    except json.JSONDecodeError:
        parsed_output = {
            "exchange_id": row["exchange_id"],
            "ma_relevance": None,
            "reason": None,
            "risk_categories": None,
            "evidence_turns": None,
            "confidence": None,
            "parse_success": False,
            "raw_output": raw_output,
        }

    pilot_results.append(parsed_output)

pilot_results_df = pd.DataFrame(pilot_results)

display(pilot_results_df)

,exchange_id,ma_relevance,reason,risk_categories,evidence_turns,confidence,parse_success,raw_output
0,JPM_2024-Q2_QA08,not_relevant,The exchange discusses non-interest-bearing de...,[],[],high,True,"{\n ""exchange_id"": ""JPM_2024-Q2_QA08"",\n ""ma..."
1,JPM_2023-Q3_QA08,relevant,The exchange discusses tech spend related to i...,"[integration_execution, operational_resilience...","[5, 6, 7]",high,True,"{\n ""exchange_id"": ""JPM_2023-Q3_QA08"",\n ""ma..."
2,JPM_2024-Q1_QA11,relevant,"The exchange discusses integration expenses, c...","[integration_execution, costs_synergies_profit...",[1],high,True,"{\n ""exchange_id"": ""JPM_2024-Q1_QA11"",\n ""ma..."
3,JPM_2023-Q4_QA03,relevant,The exchange discusses deposit migration and r...,"[integration_execution, client_franchise, capi...",[1],high,True,"{\n ""exchange_id"": ""JPM_2023-Q4_QA03"",\n ""ma..."
4,JPM_2023-Q2_QA10,not_relevant,"The exchange discusses asset sensitivity, depo...",[],[],high,True,"{\n ""exchange_id"": ""JPM_2023-Q2_QA10"",\n ""ma..."
5,UBS_2024-Q3_QA06,relevant,The exchange discusses the integration of the ...,"[integration_execution, capital_liquidity, wor...","[1, 2]",high,True,"{\n ""exchange_id"": ""UBS_2024-Q3_QA06"",\n ""ma..."
6,UBS_2023-Q4_QA10,relevant,The exchange discusses the integration of Cred...,"[integration_execution, operational_resilience]",[2],high,True,"{\n ""exchange_id"": ""UBS_2023-Q4_QA10"",\n ""ma..."
7,UBS_2024-Q3_QA02,not_relevant,"The exchange discusses profitability, transact...",[],[],high,True,"{\n ""exchange_id"": ""UBS_2024-Q3_QA02"",\n ""ma..."
8,UBS_2023-Q2_QA07,relevant,The management discusses revenue cost synergie...,"[costs_synergies_profitability, capital_liquid...","[1, 2]",high,True,"{\n ""exchange_id"": ""UBS_2023-Q2_QA07"",\n ""ma..."
9,UBS_2023-Q3_QA08,relevant,The management response details the legal enti...,"[integration_execution, governance_risk_manage...","[1, 2, 3]",high,True,"{\n ""exchange_id"": ""UBS_2023-Q3_QA08"",\n ""ma..."


In [13]:
# re-load necessary libraries
import time
import pandas as pd
from tqdm.auto import tqdm

In [14]:
# confirm that all required objects exist
required_objects = [
    "ma_qa_exchanges",
    "model",
    "tokenizer",
    "RISK_TAXONOMY",
    "SYSTEM_PROMPT",
    "build_relevance_prompt",
    "generate_response",
    "parse_json_response",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "These objects are missing after the runtime restart: "
        + ", ".join(missing_objects)
    )

print("All required objects are available.")

All required objects are available.


In [15]:
# Run the relevance classifier across every retrieved Q&A exchange

def classify_exchange(row, max_attempts=2):
    prompt = build_relevance_prompt(row)

    for attempt in range(max_attempts):
        try:
            raw_output = generate_response(prompt)
            result = parse_json_response(raw_output)

            return {
                "exchange_id": row["exchange_id"],
                "ma_relevance": result.get("ma_relevance"),
                "reason": result.get("reason"),
                "risk_categories": result.get("risk_categories", []),
                "evidence_turns": result.get("evidence_turns", []),
                "confidence": result.get("confidence"),
                "parse_success": True,
                "raw_output": raw_output,
            }

        except Exception as error:
            if attempt == max_attempts - 1:
                return {
                    "exchange_id": row["exchange_id"],
                    "ma_relevance": "uncertain",
                    "reason": f"Processing error: {error}",
                    "risk_categories": [],
                    "evidence_turns": [],
                    "confidence": "low",
                    "parse_success": False,
                    "raw_output": None,
                }

            time.sleep(1)


results = []

for row in tqdm(
    ma_qa_exchanges.to_dict("records"),
    desc="Classifying exchanges",
):
    results.append(classify_exchange(row))

relevance_results_df = pd.DataFrame(results)

# Join predictions to the exchange data.
classified_exchanges = ma_qa_exchanges.merge(
    relevance_results_df,
    on="exchange_id",
    how="left",
    validate="one_to_one",
)

# Retain relevant and uncertain exchanges.
relevant_exchanges = classified_exchanges[
    classified_exchanges["ma_relevance"].isin(
        ["relevant", "uncertain"]
    )
].copy()

print("All candidate exchanges:", len(classified_exchanges))
print("Retained exchanges:", len(relevant_exchanges))
print(classified_exchanges["ma_relevance"].value_counts(dropna=False))

# Save results.
classified_exchanges.to_csv(
    "qa_relevance_results.csv",
    index=False,
)

relevant_exchanges.to_csv(
    "relevant_qa_exchanges.csv",
    index=False,
)

Classifying exchanges:   0%|          | 0/102 [00:00<?, ?it/s]

All candidate exchanges: 102
Retained exchanges: 79
ma_relevance
relevant        70
not_relevant    23
uncertain        9
Name: count, dtype: int64


In [16]:
relevance_results_df.head()

,exchange_id,ma_relevance,reason,risk_categories,evidence_turns,confidence,parse_success,raw_output
0,JPM_2023-Q1_QA04,not_relevant,"The exchange discusses reserve assumptions, un...",[],[],high,True,"{\n ""exchange_id"": ""JPM_2023-Q1_QA04"",\n ""ma..."
1,JPM_2023-Q2_QA01,relevant,The management discussion addresses deposit re...,"[integration_execution, client_franchise, cost...","[1, 2, 3]",high,True,"{\n ""exchange_id"": ""JPM_2023-Q2_QA01"",\n ""ma..."
2,JPM_2023-Q2_QA02,relevant,The exchange discusses the loss share agreemen...,"[capital_liquidity, credit_asset_quality, inte...","[1, 2, 3]",high,True,"{\n ""exchange_id"": ""JPM_2023-Q2_QA02"",\n ""ma..."
3,JPM_2023-Q2_QA03,uncertain,The exchange touches on capital requirements a...,[],[],high,True,"{\n ""exchange_id"": ""JPM_2023-Q2_QA03"",\n ""ma..."
4,JPM_2023-Q2_QA04,uncertain,The transcript touches on loan book valuation ...,[],[],high,True,"{\n ""exchange_id"": ""JPM_2023-Q2_QA04"",\n ""ma..."


In [17]:
# rename the folder in Google Drive
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/boe-data")

OLD_DIR = BASE_DIR / "llm_analysis_checkpoints"
NEW_DIR = BASE_DIR / "inessa_llm_analysis_checkpoint"

if OLD_DIR.exists() and not NEW_DIR.exists():
    OLD_DIR.rename(NEW_DIR)
    print("Renamed checkpoint folder.")

elif OLD_DIR.exists() and NEW_DIR.exists():
    raise RuntimeError(
        "Both old and new folders exist. Do not continue until "
        "their contents have been checked."
    )

elif not NEW_DIR.exists():
    NEW_DIR.mkdir(parents=True)
    print("Created new checkpoint folder.")

CHECKPOINT_DIR = NEW_DIR

print("Checkpoint directory:", CHECKPOINT_DIR)

Renamed checkpoint folder.
Checkpoint directory: /content/drive/MyDrive/boe-data/inessa_llm_analysis_checkpoint


In [18]:
# save the results of the classifier to the Google Drive folder
from datetime import datetime
import json

drive.mount("/content/drive", force_remount=False)

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/boe-data/inessa_llm_analysis_checkpoints"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Complete working copies: preserve Python lists and data types.
relevance_results_df.to_pickle(
    CHECKPOINT_DIR / "relevance_results.pkl"
)

classified_exchanges.to_pickle(
    CHECKPOINT_DIR / "classified_exchanges.pkl"
)

relevant_exchanges.to_pickle(
    CHECKPOINT_DIR / "relevant_exchanges.pkl"
)

# CSV copies: convenient for viewing and sharing.
relevance_results_df.to_csv(
    CHECKPOINT_DIR / "relevance_results.csv",
    index=False,
)

classified_exchanges.to_csv(
    CHECKPOINT_DIR / "classified_exchanges.csv",
    index=False,
)

relevant_exchanges.to_csv(
    CHECKPOINT_DIR / "relevant_exchanges.csv",
    index=False,
)

# Record which model produced the results.
run_metadata = {
    "saved_at": datetime.now().isoformat(),
    "model_name": MODEL_NAME,
    "candidate_exchanges": len(classified_exchanges),
    "retained_exchanges": len(relevant_exchanges),
    "successful_parses": int(
        relevance_results_df["parse_success"].sum()
    ),
    "failed_parses": int(
        (~relevance_results_df["parse_success"]).sum()
    ),
}

with open(
    CHECKPOINT_DIR / "classification_run_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(run_metadata, file, indent=2)

print("Results saved successfully to:")
print(CHECKPOINT_DIR)

print("\nFiles:")
for path in sorted(CHECKPOINT_DIR.iterdir()):
    print("-", path.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results saved successfully to:
/content/drive/MyDrive/boe-data/inessa_llm_analysis_checkpoints

Files:
- classification_run_metadata.json
- classified_exchanges.csv
- classified_exchanges.pkl
- relevance_results.csv
- relevance_results.pkl
- relevant_exchanges.csv
- relevant_exchanges.pkl


In [19]:
# verify the saved master file
verification_df = pd.read_pickle(
    CHECKPOINT_DIR / "classified_exchanges.pkl"
)

print("Saved rows:", len(verification_df))
print(
    verification_df["ma_relevance"]
    .value_counts(dropna=False)
)

Saved rows: 102
ma_relevance
relevant        70
not_relevant    23
uncertain        9
Name: count, dtype: int64


# Use LLM to query the MA-related passages

In [26]:
for folder_name in [
    "inessa_llm_analysis_checkpoints",
    "inessa_llm_analysis_checkpoint",
]:
    folder = DATA / folder_name
    print(f"\n{folder_name}:")

    if folder.exists():
        for file in sorted(folder.iterdir()):
            print(" -", file.name)


inessa_llm_analysis_checkpoints:
 - classification_run_metadata.json
 - classified_exchanges.csv
 - classified_exchanges.pkl
 - relevance_results.csv
 - relevance_results.pkl
 - relevant_exchanges.csv
 - relevant_exchanges.pkl

inessa_llm_analysis_checkpoint:
 - ma_prepared_remarks.pkl
 - ma_qa_exchanges.pkl
 - ma_qa_turns.pkl
 - retrieval_coverage.pkl


In [27]:
import shutil

source = DATA / "inessa_llm_analysis_checkpoint"
destination = DATA / "inessa_llm_analysis_checkpoints"

destination.mkdir(parents=True, exist_ok=True)

for file in source.iterdir():
    target = destination / file.name

    if not target.exists():
        shutil.copy2(file, target)

CHECKPOINT_DIR = destination

print("Canonical folder:", CHECKPOINT_DIR)

Canonical folder: /content/drive/MyDrive/boe-data/inessa_llm_analysis_checkpoints


In [29]:
old_folder = DATA / "inessa_llm_analysis_checkpoint"

if old_folder.exists() and old_folder.name == "inessa_llm_analysis_checkpoint":
    shutil.rmtree(old_folder)
    print("Deleted:", old_folder)
else:
    print("Folder not found or path is incorrect.")

Deleted: /content/drive/MyDrive/boe-data/inessa_llm_analysis_checkpoint


In [28]:
ma_qa_turns = pd.read_pickle(
    CHECKPOINT_DIR / "ma_qa_turns.pkl"
)

relevant_exchanges = pd.read_pickle(
    CHECKPOINT_DIR / "relevant_exchanges.pkl"
)

In [31]:
#------------------------------------------------------------
# aggregate MA-related passages by bank and transacton stage
#------------------------------------------------------------
# Reload transaction phases
ma_qa_turns = pd.read_pickle(
    DATA / "inessa_llm_analysis_checkpoints" / "ma_qa_turns.pkl"
)

phase_columns = [
    "transaction_phase",
    "transaction_period",
    "relative_quarter",
]

phase_lookup = (
    ma_qa_turns[
        ["bank", "quarter"] + phase_columns
    ]
    .drop_duplicates(["bank", "quarter"])
)

# Prevent duplicate columns if this cell is rerun
relevant_exchanges = relevant_exchanges.drop(
    columns=[
        column for column in phase_columns
        if column in relevant_exchanges.columns
    ]
)

relevant_exchanges = relevant_exchanges.merge(
    phase_lookup,
    on=["bank", "quarter"],
    how="left",
    validate="many_to_one",
)

assert relevant_exchanges["transaction_phase"].notna().all()

# Prepare exchanges for aggregation
relevant_exchanges = relevant_exchanges.sort_values(
    ["bank", "call_date", "first_position"]
).copy()

relevant_exchanges["exchange_block"] = (
    "EXCHANGE ID: " + relevant_exchanges["exchange_id"]
    + "\nQUARTER: " + relevant_exchanges["quarter"].astype(str)
    + "\n" + relevant_exchanges["llm_input_text"]
)


def combine_unique(series):
    return list(pd.unique(series.dropna()))


def combine_categories(series):
    categories = set()

    for values in series.dropna():
        if isinstance(values, list):
            categories.update(values)

    return sorted(categories)


# Aggregate by bank and transaction phase
stage_ma_exchanges = (
    relevant_exchanges
    .groupby(
        ["bank", "transaction_phase"],
        observed=True,
        as_index=False,
    )
    .agg(
        transaction_name=("transaction_name", "first"),
        quarters=("quarter", combine_unique),
        first_call_date=("call_date", "min"),
        last_call_date=("call_date", "max"),
        number_of_exchanges=("exchange_id", "nunique"),
        exchange_ids=("exchange_id", list),
        preliminary_risk_categories=(
            "risk_categories",
            combine_categories,
        ),
        stage_transcript=(
            "exchange_block",
            lambda values: "\n\n".join(values),
        ),
    )
    .sort_values(["bank", "first_call_date"])
    .reset_index(drop=True)
)

stage_ma_exchanges["estimated_tokens"] = (
    stage_ma_exchanges["stage_transcript"].str.len() / 4
).round().astype(int)

# Save permanently
stage_ma_exchanges.to_pickle(
    DATA
    / "inessa_llm_analysis_checkpoints"
    / "stage_ma_exchanges.pkl"
)

display(stage_ma_exchanges)

,bank,transaction_phase,transaction_name,quarters,first_call_date,last_call_date,number_of_exchanges,exchange_ids,preliminary_risk_categories,stage_transcript,estimated_tokens
0,JPM,3. Initial consolidation (0–6 months),JPMorgan–First Republic,"[2023-Q2, 2023-Q3]",2023-05-01,2023-10-13,13,"[JPM_2023-Q2_QA01, JPM_2023-Q2_QA02, JPM_2023-...","[capital_liquidity, client_franchise, complexi...",EXCHANGE ID: JPM_2023-Q2_QA01\nQUARTER: 2023-Q...,16783
1,JPM,4. Early integration (6–12 months),JPMorgan–First Republic,"[2023-Q4, 2024-Q1]",2024-01-12,2024-04-12,4,"[JPM_2023-Q4_QA03, JPM_2023-Q4_QA10, JPM_2024-...","[capital_liquidity, client_franchise, costs_sy...",EXCHANGE ID: JPM_2023-Q4_QA03\nQUARTER: 2023-Q...,3748
2,JPM,5. Ongoing integration (12–24 months),JPMorgan–First Republic,[2024-Q3],2024-10-11,2024-10-11,1,[JPM_2024-Q3_QA10],"[capital_liquidity, client_franchise, costs_sy...",EXCHANGE ID: JPM_2024-Q3_QA10\nQUARTER: 2024-Q...,1320
3,UBS,2. Announced / pre-completion,UBS–Credit Suisse,[2023-Q1],2023-04-25,2023-04-25,11,"[UBS_2023-Q1_QA01, UBS_2023-Q1_QA02, UBS_2023-...","[capital_liquidity, client_franchise, complexi...",EXCHANGE ID: UBS_2023-Q1_QA01\nQUARTER: 2023-Q...,7741
4,UBS,3. Initial consolidation (0–6 months),UBS–Credit Suisse,"[2023-Q2, 2023-Q3]",2023-08-31,2023-11-07,20,"[UBS_2023-Q2_QA01, UBS_2023-Q2_QA02, UBS_2023-...","[capital_liquidity, client_franchise, costs_sy...",EXCHANGE ID: UBS_2023-Q2_QA01\nQUARTER: 2023-Q...,15842
5,UBS,4. Early integration (6–12 months),UBS–Credit Suisse,"[2023-Q4, 2024-Q1]",2024-02-06,2024-05-07,16,"[UBS_2023-Q4_QA01, UBS_2023-Q4_QA03, UBS_2023-...","[capital_liquidity, client_franchise, costs_sy...",EXCHANGE ID: UBS_2023-Q4_QA01\nQUARTER: 2023-Q...,14570
6,UBS,5. Ongoing integration (12–24 months),UBS–Credit Suisse,"[2024-Q2, 2024-Q3, 2024-Q4]",2024-08-14,2025-02-04,14,"[UBS_2024-Q2_QA01, UBS_2024-Q2_QA02, UBS_2024-...","[capital_liquidity, client_franchise, costs_sy...",EXCHANGE ID: UBS_2024-Q2_QA01\nQUARTER: 2024-Q...,12391


70 exchanges have been reduced to 7 stage-level LLM calls.
The largest prompt is about 16,783 tokens, within Qwen's context capacity.

In [32]:
#-----------------------------------------------------------------------
# run one prompt per row to extract ma-related insights and risk signals
#-----------------------------------------------------------------------
STAGE_SYSTEM_PROMPT = """
You are analysing bank M&A earnings-call passages for prudential supervision.

Use only the supplied transcript. Do not use external knowledge or invent facts.
Distinguish analyst concerns from management statements.
Every conclusion must cite supporting exchange IDs.
Return valid JSON only.
"""

def build_stage_prompt(row):
    return f"""
Analyse the following transaction stage.

Bank: {row['bank']}
Transaction: {row['transaction_name']}
Transaction phase: {row['transaction_phase']}
Quarters included: {row['quarters']}
Number of Q&A exchanges: {row['number_of_exchanges']}

Transcript:
--- START ---
{row['stage_transcript']}
--- END ---

Identify no more than five material risks.

Score the following indicators from 0 to 3:

- analyst_attention:
  0 = absent, 1 = isolated, 2 = repeated, 3 = persistent/intense

- management_response_gap:
  0 = fully answered, 1 = mostly answered, 2 = partly answered,
  3 = largely unanswered or evasive

- execution_pressure:
  0 = none, 1 = limited, 2 = material, 3 = severe or escalating

- uncertainty:
  0 = none, 1 = limited, 2 = material, 3 = high

- quantitative_support:
  0 = no quantitative evidence, 1 = limited, 2 = moderate,
  3 = strong and specific evidence

Return exactly this JSON structure:

{{
  "bank": "{row['bank']}",
  "transaction_phase": "{row['transaction_phase']}",
  "risk_profiles": [
    {{
      "risk_category": "one permitted taxonomy category",
      "specific_issue": "concise description",
      "analyst_concern": "concise description",
      "management_position": "concise description",
      "direction_within_stage": "emerging, intensifying, stable, diminishing, resolved, or unclear",
      "analyst_attention": 0,
      "management_response_gap": 0,
      "execution_pressure": 0,
      "uncertainty": 0,
      "quantitative_support": 0,
      "metrics_targets_or_deadlines": [],
      "supporting_exchange_ids": []
    }}
  ],
  "stage_indicators": {{
    "repeated_analyst_concerns": [],
    "partially_or_unanswered_concerns": [],
    "delays_or_milestone_slippage": [],
    "management_analyst_divergence": [],
    "forward_looking_warning_signals": [],
    "evidence_of_resolution_or_improvement": []
  }},
  "overall_stage_summary": "maximum three sentences"
}}

Permitted risk categories:
{format_taxonomy(RISK_TAXONOMY)}

Use empty lists where no evidence exists.
""".strip()

In [34]:
def generate_stage_response(prompt, max_new_tokens=1600):
    messages = [
        {"role": "system", "content": STAGE_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    input_length = model_inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output_ids[0, input_length:],
        skip_special_tokens=True,
    ).strip()


CHECKPOINT_DIR = DATA / "inessa_llm_analysis_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = CHECKPOINT_DIR / "stage_analysis_checkpoint.json"

# Resume previous progress if available
if checkpoint_path.exists():
    with open(checkpoint_path, encoding="utf-8") as file:
        stage_results = json.load(file)
else:
    stage_results = []

completed = {
    (result["bank"], result["transaction_phase"])
    for result in stage_results
}


for row in tqdm(
    stage_ma_exchanges.to_dict("records"),
    desc="Analysing transaction stages",
):
    stage_key = (
        row["bank"],
        str(row["transaction_phase"]),
    )

    if stage_key in completed:
        continue

    prompt = build_stage_prompt(row)

    try:
        raw_output = generate_stage_response(prompt)
        result = parse_json_response(raw_output)

        result["parse_success"] = True
        result["raw_output"] = raw_output

    except Exception as error:
        result = {
            "bank": row["bank"],
            "transaction_phase": str(row["transaction_phase"]),
            "risk_profiles": [],
            "stage_indicators": {},
            "overall_stage_summary": None,
            "parse_success": False,
            "error": str(error),
            "raw_output": (
                raw_output if "raw_output" in locals() else None
            ),
        }

    stage_results.append(result)

    # Save after every stage
    with open(
        checkpoint_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            stage_results,
            file,
            indent=2,
            ensure_ascii=False,
        )


stage_results_df = pd.DataFrame(stage_results)

stage_results_df.to_pickle(
    CHECKPOINT_DIR / "stage_analysis_results.pkl"
)

stage_results_df.to_csv(
    CHECKPOINT_DIR / "stage_analysis_results.csv",
    index=False,
)


# Print results
for result in stage_results:
    print("=" * 100)
    print(
        result["bank"],
        "|",
        result["transaction_phase"],
    )
    print(json.dumps(result, indent=2, ensure_ascii=False))

print("\nSaved to:", CHECKPOINT_DIR)

Analysing transaction stages:   0%|          | 0/7 [00:00<?, ?it/s]

JPM | 3. Initial consolidation (0–6 months)
{
  "bank": "JPM",
  "transaction_phase": "3. Initial consolidation (0–6 months)",
  "risk_profiles": [],
  "stage_indicators": {},
  "overall_stage_summary": null,
  "parse_success": false,
  "error": "Expecting value: line 117 column 30 (char 5561)",
  "raw_output": "{\n  \"bank\": \"JPM\",\n  \"transaction_phase\": \"3. Initial consolidation (0–6 months)\",\n  \"risk_profiles\": [\n    {\n      \"risk_category\": \"client_franchise\",\n      \"specific_issue\": \"Deposit attrition and client retention risks post-acquisition\",\n      \"analyst_concern\": \"Analysts question the level of deposit retention and advisor migration, asking about attrition assumptions and expense synergies in wealth management.\",\n      \"management_position\": \"Management expresses caution, stating they are 'very eager' to retain clients and 'fight hard' to keep them, but acknowledges uncertainty and will provide updates at Investor Day and future calls.\",\n 

### **CHANGE RUNTIME TYPE TO CPU TO PRESERVE RESOURCES**

In [1]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import json

drive.mount("/content/drive")

DATA = Path("/content/drive/MyDrive/boe-data")
CHECKPOINT_DIR = DATA / "inessa_llm_analysis_checkpoints"

print("CPU runtime ready.")

Mounted at /content/drive
CPU runtime ready.


In [2]:
import json
import pandas as pd

CHECKPOINT_DIR = DATA / "inessa_llm_analysis_checkpoints"

# Reload results if necessary
with open(
    CHECKPOINT_DIR / "stage_analysis_checkpoint.json",
    encoding="utf-8",
) as file:
    stage_results = json.load(file)

records = []

for stage in stage_results:
    if not stage.get("parse_success"):
        continue

    for risk in stage.get("risk_profiles", []):
        records.append({
            "bank": stage["bank"],
            "transaction_phase": stage["transaction_phase"],
            "risk_category": risk.get("risk_category"),
            "specific_issue": risk.get("specific_issue"),
            "direction": risk.get("direction_within_stage"),
            "analyst_attention": risk.get("analyst_attention", 0),
            "management_response_gap": risk.get(
                "management_response_gap", 0
            ),
            "execution_pressure": risk.get("execution_pressure", 0),
            "uncertainty": risk.get("uncertainty", 0),
            "quantitative_support": risk.get(
                "quantitative_support", 0
            ),
            "supporting_exchange_ids": ", ".join(
                risk.get("supporting_exchange_ids", [])
            ),
        })

indicator_table = pd.DataFrame(records)

# Reusable supervisory priority score: 0–100
indicator_table["priority_score"] = (
    (
        indicator_table["analyst_attention"]
        + indicator_table["management_response_gap"]
        + indicator_table["execution_pressure"]
        + indicator_table["uncertainty"]
        + (3 - indicator_table["quantitative_support"])
    )
    / 15
    * 100
).round(1)

indicator_table["supervisory_flag"] = (
    (indicator_table["priority_score"] >= 60)
    | (
        indicator_table["direction"].isin(
            ["emerging", "intensifying"]
        )
        & (indicator_table["management_response_gap"] >= 2)
    )
)

# Keep the three highest-priority risks per stage
top_indicators = (
    indicator_table
    .sort_values(
        ["bank", "transaction_phase", "priority_score"],
        ascending=[True, True, False],
    )
    .groupby(
        ["bank", "transaction_phase"],
        observed=True,
    )
    .head(3)
    .reset_index(drop=True)
)

display(top_indicators)

# Save to Google Drive
top_indicators.to_csv(
    CHECKPOINT_DIR / "top_supervisory_indicators.csv",
    index=False,
)

with pd.ExcelWriter(
    CHECKPOINT_DIR / "supervisory_indicators.xlsx",
    engine="openpyxl",
) as writer:
    top_indicators.to_excel(
        writer,
        sheet_name="Top indicators",
        index=False,
    )
    indicator_table.to_excel(
        writer,
        sheet_name="All risks",
        index=False,
    )

print("Files saved to:", CHECKPOINT_DIR)

,bank,transaction_phase,risk_category,specific_issue,direction,analyst_attention,management_response_gap,execution_pressure,uncertainty,quantitative_support,supporting_exchange_ids,priority_score,supervisory_flag
0,JPM,4. Early integration (6–12 months),credit_asset_quality,Sensitivity of First Republic's NII to rate cu...,emerging,2,3,1,3,1,JPM_2023-Q4_QA10,73.3,True
1,JPM,4. Early integration (6–12 months),costs_synergies_profitability,Integration cost trajectory and accretion timi...,intensifying,2,1,2,2,2,"JPM_2023-Q4_QA10, JPM_2024-Q1_QA11",53.3,False
2,JPM,4. Early integration (6–12 months),integration_execution,Integration timeline and progress on cost-savi...,intensifying,2,1,2,2,2,JPM_2024-Q1_QA11,53.3,False
3,JPM,5. Ongoing integration (12–24 months),client_franchise,Pressure on deposit margins due to falling pol...,stable,2,0,1,2,2,JPM_2024-Q3_QA10,40.0,False
4,JPM,5. Ongoing integration (12–24 months),capital_liquidity,Potential funding pressure from declining depo...,stable,2,0,1,2,2,JPM_2024-Q3_QA10,40.0,False
5,UBS,2. Announced / pre-completion,integration_execution,"Uncertainty around integration model, includin...",intensifying,3,2,3,3,1,"UBS_2023-Q1_QA07, UBS_2023-Q1_QA04",86.7,True
6,UBS,2. Announced / pre-completion,costs_synergies_profitability,Revenue attrition and cost-saving assumptions ...,emerging,2,1,2,2,2,"UBS_2023-Q1_QA09, UBS_2023-Q1_QA02",53.3,False
7,UBS,2. Announced / pre-completion,governance_risk_management,Uncertainty around loss-sharing guarantees and...,emerging,2,1,1,2,1,UBS_2023-Q1_QA01,53.3,False


Files saved to: /content/drive/MyDrive/boe-data/inessa_llm_analysis_checkpoints
